# Multi-year causal model experiment
This notebook explains and reproduces the research-only comparison. The 24-hour source delay is an assumption, not verified publication timing. No model is promoted from this notebook.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root / 'src'))
from gridtoev.history_model import build_causal_history_features, model_feature_columns
from gridtoev.history_experiment import evaluate_history_experiment
contract = json.loads((root / 'config/multi_year_causal_experiment.v1.json').read_text())
history = pd.read_csv(root / contract['source_path'])
print(f'Source intervals: {len(history):,}')

## Audit the feature cutoff
Each issue uses only a historical snapshot at least 24 hours old. Targets are joined at exactly issue + 30 or + 60 minutes; no row-shift approximation is used.

In [ ]:
features = build_causal_history_features(history, observation_delay_hours=contract['observation_delay_hours'])
assert (features['feature_timestamp_utc'] <= features['latest_eligible_observation_utc']).all()
assert not features.duplicated(['issue_timestamp_utc', 'forecast_horizon_minutes']).any()
print(f'Model rows: {len(features):,}; numeric features: {len(model_feature_columns(features))}')
features[['issue_timestamp_utc', 'feature_timestamp_utc', 'target_timestamp_utc', 'forecast_horizon_minutes', 'dispatch_down_mwh']].head()

## Predeclared monthly evaluation
May–July 2026 are development folds. August is held back unless the development gate passes. Running the next cell fits six small horizon-specific models and takes some time.

In [ ]:
report = evaluate_history_experiment(history, contract)
summary = pd.DataFrame(report['development']['folds'])[['month_utc', 'baseline_mae_mwh', 'candidate_mae_mwh', 'mae_improvement_fraction']]
display(summary)
print('Development gate:', report['development']['checks'])
print('August accessed:', report['sealed_final']['accessed'])
print('Release approved:', report['release']['candidate_approved'])

The committed machine-readable report includes the source checksum. A 5.83% development gain did not satisfy the 15% and fold-stability gates. Do not compare that figure directly to the deployed v1.1.0 model: it uses a different feature population and comparator.